# 03 — Standalone Predictor
Run this cell on any image after training is complete.

**Features:**
- Loads all models from `pipeline_v2/models/`
- Extracts 1280 (EfficientNet) + 30 (handcrafted) = **1310-d** feature vector
- Fruit confidence gate: if top-1 probability ≥ 0.70 → use it; else evaluate top-2 with validity check
- Returns: fruit, freshness %, confidence %, low-confidence flag

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import numpy as np
import cv2
import joblib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from skimage.feature import graycomatrix, graycoprops
from scipy.stats import entropy
import tensorflow as tf

# ── CONFIG ────────────────────────────────────────────────────────────
IMAGE_PATH           = "test_img/fresh_cucumber3.jpg"  # ← change this
PIPELINE_DIR         = "pipeline_v2"
FRUIT_CONF_THRESHOLD = 0.70   # tune if needed
# ─────────────────────────────────────────────────────────────────────


## Feature extractor  
**Do not modify** — must match the extractor used in `00_extract.ipynb`.

In [ ]:
def _extract_handcrafted(img_bgr) -> np.ndarray:
    """30 features: RGB(6)+HSV(6)+LAB(6)+Texture(5)+Shape(6)+Dark(1)."""
    img = cv2.resize(img_bgr, (224, 224)).astype(np.float32)
    b, g, r = cv2.split(img)
    rgb = [r.mean(), g.mean(), b.mean(), r.std(), g.std(), b.std()]
    hsv     = cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_BGR2HSV)
    h, s, v = (hsv[:, :, i].astype(np.float32) for i in range(3))
    h_rad   = h * (2 * np.pi / 180)
    cx, sx  = np.cos(h_rad).mean(), np.sin(h_rad).mean()
    hsv_f   = [float(np.arctan2(sx, cx)), float(s.mean()), float(v.mean()),
               float(np.sqrt(-2 * np.log(np.sqrt(cx**2 + sx**2) + 1e-6))),
               float(s.std()), float(v.std())]
    lab     = cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_BGR2LAB)
    L, a, b2 = (lab[:, :, i].astype(np.float32) for i in range(3))
    lab_f   = [L.mean(), L.std(), a.mean(), a.std(), b2.mean(), b2.std()]
    gray    = cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_BGR2GRAY)
    lap     = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    glcm    = graycomatrix((gray // 8).astype(np.uint8), [1], [0],
                           levels=32, symmetric=True, normed=True)
    hist, _ = np.histogram(gray, bins=256, range=(0, 255), density=True)
    tex_f   = [lap,
               float(graycoprops(glcm, "contrast")[0, 0]),
               float(graycoprops(glcm, "energy")[0, 0]),
               float(graycoprops(glcm, "homogeneity")[0, 0]),
               float(entropy(hist + 1e-6))]
    _, th   = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if cnts:
        c     = max(cnts, key=cv2.contourArea)
        area  = float(cv2.contourArea(c))
        peri  = float(cv2.arcLength(c, True))
        hull_a = float(cv2.contourArea(cv2.convexHull(c)))
        x_, y_, w_, h_ = cv2.boundingRect(c)
        shp_f = [area, peri, 4*np.pi*area/(peri*peri+1e-6),
                 area/(hull_a+1e-6), w_/(h_+1e-6), area/(w_*h_+1e-6)]
    else:
        shp_f = [0.0] * 6
    feats = rgb + hsv_f + lab_f + tex_f + shp_f + [float((gray < 50).mean())]
    assert len(feats) == 30, f"Handcrafted count error: {len(feats)}"
    return np.array(feats, dtype=np.float32)


## Load models  *(once per session)*

In [ ]:
print("[1/3] Loading models ...", end=" ", flush=True)
m = f"{PIPELINE_DIR}/models"
a = f"{PIPELINE_DIR}/artifacts"

global_scaler = joblib.load(f"{m}/scaler.pkl")
scaler_fruit  = joblib.load(f"{m}/scaler_fruit.pkl")
scaler_fresh  = joblib.load(f"{m}/scaler_fresh.pkl")
rfe_fruit     = joblib.load(f"{m}/rfe_fruit.pkl")
rfe_fresh     = joblib.load(f"{m}/rfe_fresh.pkl")
svm_fruit     = joblib.load(f"{m}/svm_fruit.pkl")
dasfs_dict    = joblib.load(f"{m}/dasfs.pkl")
knn_bundle    = joblib.load(f"{m}/knn_dict.pkl")
knn_dict      = knn_bundle["knn_dict"]
tau_dict      = knn_bundle["tau_dict"]
top300_idx    = np.load(f"{a}/top300_cmi.npy")
print("done")

print("[2/3] Loading EfficientNetB0 ...", end=" ", flush=True)
backbone   = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", pooling="avg"
)
preprocess = tf.keras.applications.efficientnet.preprocess_input
print("done")


## Predict function  
Includes fruit confidence gate + top-2 validity-checked fallback.

In [ ]:
def _freshness_for_fruit(xf_vec, fruit):
    """Compute DASFS score, DASFS confidence, KNN support for one fruit candidate.
    Returns (score, dasfs_conf, knn_sup, proj) or None if fruit not in models.
    """
    if fruit not in dasfs_dict or fruit not in knn_dict:
        return None
    d      = dasfs_dict[fruit]
    proj   = float(xf_vec @ d["axis"])
    p_f, p_r, spread = d["p_fresh"], d["p_rotten"], d["spread"]
    score  = float(np.clip((p_r - proj) / (p_r - p_f + 1e-8), 0.0, 1.0))
    mid    = (p_f + p_r) / 2.0
    d_conf = float(np.clip(1.0 - np.exp(-((proj - mid)**2) /
                                          (2.0 * spread**2 + 1e-8)), 0.0, 1.0))
    k_dist = float(knn_dict[fruit].kneighbors(xf_vec.reshape(1, -1))[0].mean())
    knn_sup = float(np.exp(-k_dist / (tau_dict[fruit] + 1e-8)))
    return score, d_conf, knn_sup, proj


def _valid_candidate(fruit, proj):
    """Accept a candidate only if proj falls within the fruit's full distribution range."""
    d = dasfs_dict[fruit]
    spread = d["spread"]
    return (proj >= d["p_fresh"] - 2 * spread) and (proj <= d["p_rotten"] + 2 * spread)


def predict(image_path: str) -> dict:
    """End-to-end freshness prediction with fruit confidence gate.

    Returns
    -------
    dict with keys:
        fruit            : str   — predicted fruit type
        label            : str   — freshness category
        freshness_score  : float — 0–100
        dasfs_score      : float — 0–100
        knn_support      : float — 0–100
        confidence       : float — 0–100
        low_conf_flag    : bool  — True if top-2 fallback was used
    """
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(f"Cannot read: {image_path}")

    # ── Extract 1310-d feature vector ──
    img_rgb   = cv2.resize(img, (224, 224))
    emb       = backbone.predict(
        preprocess(np.expand_dims(img_rgb.astype(np.float32), 0)), verbose=0
    ).flatten()                                    # (1280,)
    hand      = _extract_handcrafted(img)          # (30,)
    raw_feat  = np.concatenate([emb, hand]).reshape(1, -1)  # (1, 1310)

    assert raw_feat.shape[1] == 1310, f"Feature dim mismatch: {raw_feat.shape[1]}"

    # ── Scale ──
    x_sc = global_scaler.transform(raw_feat)       # (1, 1310)

    # ── Fruit branch ──
    x_fruit = scaler_fruit.transform(rfe_fruit.transform(x_sc))

    # ── Freshness branch (independent of fruit choice) ──
    x_fresh = scaler_fresh.transform(
        rfe_fresh.transform(x_sc[:, top300_idx])
    )
    xf_vec  = x_fresh[0]

    # ── Fruit confidence gate ──
    probs      = svm_fruit.predict_proba(x_fruit)[0]
    sorted_idx = np.argsort(probs)[::-1]
    classes    = svm_fruit.classes_
    top1_fruit = str(classes[sorted_idx[0]])
    top1_prob  = float(probs[sorted_idx[0]])

    low_conf_flag = False

    if top1_prob >= FRUIT_CONF_THRESHOLD:
        fruit = top1_fruit
    else:
        # ── Top-2 fallback with validity check ──
        low_conf_flag = True
        candidates    = [str(classes[sorted_idx[i]]) for i in range(min(2, len(classes)))]
        best_fruit    = None
        best_conf     = -1.0

        for cand in candidates:
            res = _freshness_for_fruit(xf_vec, cand)
            if res is None:
                continue
            _, d_conf, knn_sup, proj = res
            if not _valid_candidate(cand, proj):
                continue
            final_conf = 0.6 * d_conf + 0.4 * knn_sup
            if final_conf > best_conf:
                best_conf  = final_conf
                best_fruit = cand

        fruit = best_fruit if best_fruit is not None else top1_fruit

    # ── Freshness scoring for chosen fruit ──
    if fruit not in dasfs_dict or fruit not in knn_dict:
        return {"fruit": fruit, "label": "Unknown",
                "freshness_score": -1.0, "confidence": 0.0,
                "low_conf_flag": low_conf_flag}

    res        = _freshness_for_fruit(xf_vec, fruit)
    score, d_conf, knn_sup, _ = res

    freshness  = round(score * 100.0, 2)
    confidence = round((0.6 * d_conf + 0.4 * knn_sup) * 100.0, 1)

    if   freshness > 75: label = "Very Fresh"
    elif freshness > 50: label = "Fresh"
    elif freshness > 25: label = "Slightly Degraded"
    else:                label = "Rotten"

    return {
        "fruit"          : fruit,
        "label"          : label,
        "freshness_score": freshness,
        "dasfs_score"    : round(score * 100, 2),
        "knn_support"    : round(knn_sup * 100, 2),
        "confidence"     : confidence,
        "low_conf_flag"  : low_conf_flag,
    }


print("predict() ready.")
print(f"Fruit confidence threshold: {FRUIT_CONF_THRESHOLD}")
print("Call: predict('path/to/image.jpg')")


## Run prediction + display

In [ ]:
print(f"[3/3] Running inference on: {IMAGE_PATH}", flush=True)
r = predict(IMAGE_PATH)

# ── Colour by grade ──
lc = {"Very Fresh": "#2ca02c", "Fresh": "#74c476",
      "Slightly Degraded": "#fd8d3c", "Rotten": "#d62728"}.get(r["label"], "#888")

img     = cv2.imread(IMAGE_PATH)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

fig = plt.figure(figsize=(11, 4.2), facecolor="white")
gs  = fig.add_gridspec(1, 2, width_ratios=[1, 1.6], wspace=0.04)

ax_img = fig.add_subplot(gs[0])
ax_img.imshow(img_rgb); ax_img.axis("off")
ax_img.set_title(Path(IMAGE_PATH).name, fontsize=9, color="#555", pad=4, style="italic")

ax = fig.add_subplot(gs[1])
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")

# Fruit chip
ax.text(0.04, 0.93, "Fruit detected", fontsize=9, color="#888", va="top")
ax.text(0.04, 0.82, r["fruit"], fontsize=26, fontweight="bold", color="#222", va="top")
if r["low_conf_flag"]:
    ax.text(0.04, 0.74, "⚠ low-confidence routing", fontsize=8, color="#e67e00", va="top")

# Grade badge
badge = mpatches.FancyBboxPatch((0.04, 0.60), 0.42, 0.10,
    boxstyle="round,pad=0.01", linewidth=0, facecolor=lc,
    transform=ax.transAxes, clip_on=False)
ax.add_patch(badge)
ax.text(0.25, 0.655, r["label"], fontsize=12, fontweight="bold", color="white",
        ha="center", va="center", transform=ax.transAxes)

# Freshness bar
freshness = r["freshness_score"]
ax.text(0.04, 0.54, "Freshness score", fontsize=9, color="#888", va="top")
ax.text(0.96, 0.54, f"{freshness:.1f}%", fontsize=9, color="#888", va="top", ha="right")
bar_track = mpatches.FancyBboxPatch((0.04, 0.41), 0.92, 0.06,
    boxstyle="round,pad=0.004", linewidth=0, facecolor="#eee",
    transform=ax.transAxes, clip_on=False)
ax.add_patch(bar_track)
bar_w = 0.92 * (freshness / 100)
if bar_w > 0.02:
    ax.add_patch(mpatches.FancyBboxPatch((0.04, 0.41), bar_w, 0.06,
        boxstyle="round,pad=0.004", linewidth=0, facecolor=lc,
        transform=ax.transAxes, clip_on=False))

# Sub-metrics
for i, (k, v) in enumerate([
    ("DASFS score", f"{r['dasfs_score']:.1f}%"),
    ("KNN support", f"{r['knn_support']:.1f}%"),
    ("Confidence",  f"{r['confidence']:.1f}%"),
]):
    ypos = 0.31 - i * 0.12
    ax.text(0.04, ypos, k, fontsize=9, color="#888", va="top")
    ax.text(0.96, ypos, v, fontsize=10, fontweight="bold", color="#333",
            ha="right", va="top")
    ax.axhline(ypos - 0.03, xmin=0.04, xmax=0.96, color="#eee", linewidth=0.8)

plt.tight_layout(pad=0.5)
plt.show()

print(f"\n  Fruit        : {r['fruit']}")
print(f"  Grade        : {r['label']}")
print(f"  Score        : {r['freshness_score']}%")
print(f"  Confidence   : {r['confidence']}%")
print(f"  DASFS        : {r['dasfs_score']}%   KNN: {r['knn_support']}%")
print(f"  Low-conf flag: {r['low_conf_flag']}")
